# 2_5 · 仿真遥操采集（无真机兜底线）

没有真机械臂时，用 SO-101 **仿真器**顶替 `2_3` 的真机 Leader–Follower 臂。

关键是**不换命令、只换机器人**：

```
真机： lerobot-record --robot.type=so101_follower --teleop.type=keyboard ...
仿真： lerobot-record --robot.type=so101_sim      --teleop.type=keyboard ...
```

两条命令除 `--robot.type` 之外逐字相同 —— 同一个 `lerobot-record`、同一个键盘遥操器、
同一个数据集写入器。所以单位、帧率、动作语义、字段名**不需要事后对齐**，
它们没有第二个实现，也就无处走偏。


## 1 为什么不自己写一套

这一节曾是另一套东西：自己起 ManiSkill 环境、自己接 pynput 监听键盘、用 ManiSkill 的
`RecordEpisode` 录 h5、再调 `convert_to_lerobot` 转格式。

那条**平行实现**与真机线差了四项：

| 项 | 平行实现 | 真机线 |
|---|---|---|
| 动作语义 | 归一化 ±1 的**增量** | **绝对**关节位置目标 |
| 关节单位 | 原生**弧度** | 臂关节度、夹爪 0~100 行程百分比 |
| 帧率 | 20 | **30** |
| 画面 | 128×128 | 标定的 **480×640** |

四项里任何一项都不报错，只让下游静默学错 —— 而它的注释还写着「与真机数据集逐字段一致」。

**平行实现是这些差异的根源，不是它们的表现。** 逐项去改单位、改帧率，改完仍然是两套
东西，下一次改动照样会分岔。所以整段退成一条标准命令。


## 2 依赖与参数

参数字段名与真机线的 `lerobot-record` 同名，换机器人时不用改叫法。

In [ ]:
import os
import subprocess
from pathlib import Path


In [ ]:
# 场景：换任务改这一行。三个分发场景见 so101_sim 的 README。
TASK = "SO101PickPlaceCube40-v1"
# 数据集标识与落点。与真机线同一套命名，下游分不出这批数据来自真机还是仿真。
REPO_ID = "so101_sim/teleop_cube40"
SINGLE_TASK = "pick up the cube and place it in the bin"
# 采多少集、每集多长。与真机线的默认一致。
NUM_EPISODES = 5
EPISODE_TIME_S = 20
RESET_TIME_S = 5


## 3 拼出那条命令

`--robot.discover_packages_path=so101_sim` 是 lerobot 的插件发现口：它 import 仿真包
从而完成机器人注册。lerobot 侧不需要为此改任何代码。

换成真机只动两行：`--robot.type=so101_follower` 加上 `--robot.port=/dev/tty...`。


In [ ]:
def record_cmd(root: Path) -> list[str]:
    """拼出那条标准命令。

    Args:
        root: 数据集落盘目录。

    Returns:
        可直接交给 subprocess 的命令行。
    """
    return [
        "lerobot-record",
        # ── 机器人：真机换成 --robot.type=so101_follower --robot.port=/dev/tty... 即可 ──
        "--robot.type=so101_sim",
        "--robot.discover_packages_path=so101_sim",
        f"--robot.task={TASK}",
        # ── 遥操：lerobot 自带的键盘遥操器，与真机线用的是同一个 ──
        "--teleop.type=keyboard",
        # ── 数据集：fps 不写死在这里，用真机那条线的同一个值 ──
        f"--dataset.repo_id={REPO_ID}",
        f"--dataset.root={root}",
        f"--dataset.single_task={SINGLE_TASK}",
        f"--dataset.num_episodes={NUM_EPISODES}",
        f"--dataset.episode_time_s={EPISODE_TIME_S}",
        f"--dataset.reset_time_s={RESET_TIME_S}",
        "--dataset.push_to_hub=false",
        "--display_data=true",
    ]


## 4 跑它

In [ ]:
def main() -> int:
    """跑那条命令。

    Returns:
        子进程的退出码。
    """
    root = Path(os.environ["DATASETS_ROOT"]) / "so101_sim" / "_teleop" / TASK
    cmd = record_cmd(root)
    print("键盘遥操采集，走的是驱动真机的那条命令：\n")
    print("    " + " \\\n        ".join(cmd) + "\n")
    # check=False：采集被 Ctrl-C 或 Esc 打断是正常收尾，不该当异常抛出；
    # 退出码原样透出，调用方要判就自己判。
    return subprocess.run(cmd, check=False).returncode


In [ ]:
main()


## 5 产出

数据集落 `DATASETS_ROOT/so101_sim/_teleop/<TASK>/`，是标准 `LeRobotDataset`：

| 字段 | 取值 |
|---|---|
| `action` · `observation.state` | f32×6，五个臂关节**度**、夹爪**0~100 行程百分比** |
| `action` 语义 | 绝对关节位置目标 |
| 相机 | `observation.images.top` 与 `observation.images.wrist`，480×640 |
| `fps` | 30 |
| `robot_type` | `so_follower` |

夹爪与臂关节单位不同，是因为真机就是这样：`lerobot-record` 走 `so_follower`，
而它把 gripper 写死为 `MotorNormMode.RANGE_0_100`（与 `use_degrees` 无关）。
仿真沿用同一套，不是巧合也不是妥协。

⚠️ 度数与百分比的**量级恰好撞车**（夹爪物理行程约 0~100 度），所以口径搞错时
看数值看不出来，只表现为抓取这一环学不动 —— 而抓取往往正是唯一学不会的环节。
